In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_PATH = "../data/processed/menganti_SO2_daily.csv"
OUTPUT_PATH = "../data/processed/menganti_SO2_preprocessed.csv"

df = pd.read_csv(INPUT_PATH)

df["date"] = pd.to_datetime(df["date"])
df["SO2"] = pd.to_numeric(df["SO2"], errors="coerce")

df = df.sort_values("date").reset_index(drop=True)

print("Jumlah baris :", len(df))
print("Missing SO2  :", df["SO2"].isna().sum())
print("Tanggal awal :", df["date"].min())
print("Tanggal akhir:", df["date"].max())

Jumlah baris : 366
Missing SO2  : 110
Tanggal awal : 2025-08-31 00:00:00
Tanggal akhir: 2026-08-31 00:00:00


In [2]:
Q1 = df["SO2"].quantile(0.25)
Q3 = df["SO2"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (df["SO2"] < lower_bound) |
    (df["SO2"] > upper_bound)
)

print("Q1             :", Q1)
print("Q3             :", Q3)
print("IQR            :", IQR)
print("Batas bawah    :", lower_bound)
print("Batas atas     :", upper_bound)
print("Jumlah outlier :", outlier_mask.sum())

print("\nData outlier:")
display(df.loc[outlier_mask, ["date", "SO2"]])

Q1             : -1.4578156888741702e-05
Q3             : 0.00019044577771897502
IQR            : 0.00020502393460771672
Batas bawah    : -0.0003221140588003168
Batas atas     : 0.0004979816796305502
Jumlah outlier : 19

Data outlier:


,date,SO2
5,2025-09-05,-0.000337
6,2025-09-06,0.000646
11,2025-09-11,-0.000388
18,2025-09-18,0.000621
22,2025-09-22,0.000553
27,2025-09-27,0.000654
72,2025-11-11,0.000751
83,2025-11-22,0.000655
120,2025-12-29,0.000717
219,2026-04-07,-0.000683


In [3]:
df.loc[outlier_mask, "SO2"] = np.nan

print("Missing sebelum outlier :", 110)
print("Missing setelah outlier :", df["SO2"].isna().sum())

Missing sebelum outlier : 110
Missing setelah outlier : 129


In [4]:
df = df.set_index("date")

df["SO2"] = (
    df["SO2"]
    .interpolate(method="time")
    .ffill()
    .bfill()
)

df = df.reset_index()

print("Missing setelah imputasi :", df["SO2"].isna().sum())
print("Jumlah baris             :", len(df))
print("Tanggal awal             :", df["date"].min())
print("Tanggal akhir            :", df["date"].max())

Missing setelah imputasi : 0
Jumlah baris             : 366
Tanggal awal             : 2025-08-31 00:00:00
Tanggal akhir            : 2026-08-31 00:00:00


In [5]:
df.to_csv(OUTPUT_PATH, index=False)

print(f"File tersimpan: {OUTPUT_PATH}")

File tersimpan: ../data/processed/menganti_SO2_preprocessed.csv
